# Comparative Machine-Learning Benchmark: Inhibition Efficiency Across Engineering Materials

**Purpose.** A like-for-like comparison of several regression models on a mixed categorical/numerical
experimental dataset, using a single preprocessing pipeline so that differences in score reflect the
model rather than the data handling.

**Contents.** Schema-checked loading and missing-value reporting; a `ColumnTransformer` pipeline
(one-hot for categorical predictors, scaling for numerical); cross-validated hyperparameter search over
Decision Tree, Random Forest, SVR, MLP and XGBoost; held-out performance comparison; SHAP feature
attribution; and Williams leverage/residual diagnostics to bound the applicability domain.

**Data.** Expects `../data/ML_waste_material_WL_Paper_2.xlsx`. Tables and figures are written to
`outputs_inhibition_efficiency/`.

This notebook is the applied counterpart to `01_brann_materials_degradation.ipynb`, which implements a
single regularised network from scratch in PyTorch rather than benchmarking library estimators.


In [ ]:
# Comparative Machine-Learning Workflow for Inhibition-Efficiency Prediction

This notebook is provided as a **code sample** demonstrating a reproducible comparative machine-learning workflow for a materials-degradation problem.

**Methods demonstrated:** mixed numerical/categorical preprocessing, pipelines, cross-validation, hyperparameter search, Decision Tree, Random Forest, Support Vector Regression, Multilayer Perceptron, optional XGBoost, performance comparison, and applicability-domain analysis.

> **Data availability:** The research dataset is associated with work that is not yet cleared for public release. It is therefore excluded from this application code sample. The workflow will execute when an authorised local dataset with the documented schema is supplied.

from __future__ import annotations

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

from sklearn.decomposition import PCA, TruncatedSVD

warnings.filterwarnings("ignore")

# Optional packages
HAS_XGB = True
HAS_SHAP = True

try:
    from xgboost import XGBRegressor
except Exception:
    HAS_XGB = False

try:
    import shap
except Exception:
    HAS_SHAP = False

In [ ]:
results_ad = []

for model_name, model_pipeline in fitted_models.items():
    # Extract the preprocessor from the current model's pipeline
    current_preprocessor = model_pipeline.named_steps["preprocessor"]

    # Transform X_train and X_test using the current preprocessor
    X_train_t_current = current_preprocessor.transform(X_train)
    X_test_t_current = current_preprocessor.transform(X_test)

    # Compute leverage and h_star for the current model
    leverage_train_current, leverage_test_current, h_star_current = compute_leverage(
        X_train_t_current, X_test_t_current
    )

    # Make predictions using the current model
    y_test_pred_current = model_pipeline.predict(X_test)

    # Calculate residuals for the test set
    residual_test_current = y_test.values - y_test_pred_current

    # Calculate standard deviation of training residuals (from the current model's predictions)
    y_train_pred_current = model_pipeline.predict(X_train)
    residual_train_current = y_train.values - y_train_pred_current
    std_train_current = np.std(residual_train_current, ddof=1)
    if std_train_current == 0:
        std_train_current = 1e-8 # Avoid division by zero

    # Calculate standardized residuals for the test set
    std_resid_test_current = residual_test_current / std_train_current

    # Filter test observations for applicability domain analysis
    test_leverage = leverage_test_current
    test_std_resid = std_resid_test_current

    # Calculate number of test observations outside \u00b13 residuals
    test_outliers_count = np.sum(np.abs(test_std_resid) > 3)

    # Calculate number of test observations within the applicability domain
    # (leverage <= h_star AND |standardized residual| <= 3)
    within_ad_count = np.sum(
        (test_leverage <= h_star_current) & (np.abs(test_std_resid) <= 3)
    )
    total_test_observations = len(test_leverage)
    percentage_within_ad = (within_ad_count / total_test_observations) * 100

    results_ad.append({
        "Model": model_name,
        "h_star": h_star_current,
        "Test Observations Outside \u00b13 Std Residuals": test_outliers_count,
        "Test Observations Within AD": within_ad_count,
        "Percentage Test Observations Within AD": percentage_within_ad
    })

applicability_domain_df = pd.DataFrame(results_ad)
print("\nApplicability Domain Analysis for each model:")
display(applicability_domain_df)

In [ ]:
# -----------------------------
# STEP 2: USER SETTINGS
# -----------------------------
DATA_PATH = "../data/ML_waste_material_WL_Paper_2.xlsx"   # <-- change to your file path

RANDOM_STATE = 42
TEST_SIZE = 0.30
CV_SPLITS = 5

TARGET_COL = "Inhibition Efficiency (%)"

# Predictors to keep
CATEGORICAL_COLS = [
    "Material",
    "Medium or Solution"
]

NUMERICAL_COLS = [
    "Inhibitor Concentration (g)",
    "Temperature (degK)",
    "Exposure Time (hour)"
]

# Output folders
OUTPUT_DIR = Path("outputs_inhibition_efficiency")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

for folder in [FIG_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# -----------------------------
# STEP 3: LOAD DATA
# -----------------------------
def load_data(path: str) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        raise ValueError("Use a CSV or Excel file.")

    df.columns = df.columns.str.strip()
    return df


df = load_data(DATA_PATH)
print("Initial shape:", df.shape)
print(df.head())

In [ ]:
df.head()

In [ ]:
# -----------------------------
# STEP 4: BASIC DATA CLEANING
# -----------------------------
# Remove duplicates
duplicate_count = df.duplicated().sum()
print("\nDuplicate rows:", duplicate_count)
if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)

# Keep only required columns
required_cols = CATEGORICAL_COLS + NUMERICAL_COLS + [TARGET_COL]
missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise ValueError(f"These required columns are missing from the dataset: {missing_required}")

df = df[required_cols].copy()

# Convert numerical columns and target to numeric
for col in NUMERICAL_COLS + [TARGET_COL]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Missing-value report
missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percent": (df.isnull().mean() * 100).round(2).values,
    "Dtype": df.dtypes.astype(str).values
}).sort_values("Missing_Percent", ascending=False)

print("\nMissing-value summary:")
print(missing_report)
missing_report.to_csv(TABLE_DIR / "missing_value_summary.csv", index=False)

# Drop rows with missing target
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

print("\nShape after cleaning:", df.shape)


In [ ]:
df.isna().sum()

In [ ]:
# -----------------------------------------------------
# STEP 5: HANDLE MISSING NUMERICAL VALUES (IMPUTATION)
# -----------------------------------------------------

# Impute missing numerical values with the median for 'Temperature (degK)'
# (Other numerical columns already have 0 missing values as per missing_report)
median_temperature = df['Temperature (degK)'].median()
df['Temperature (degK)'] = df['Temperature (degK)'].fillna(median_temperature)
print(f"Missing 'Temperature (degK)' values filled with median: {median_temperature}")

# Update missing-value report after imputation
missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percent": (df.isnull().mean() * 100).round(2).values,
    "Dtype": df.dtypes.astype(str).values
}).sort_values("Missing_Percent", ascending=False)

print("\nMissing-value summary after imputation:")
print(missing_report)
missing_report.to_csv(TABLE_DIR / "missing_value_summary_after_imputation.csv", index=False)

In [ ]:
df.sample(10)

In [ ]:
df["Temperature (degK)"].unique()

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
# -----------------------------
# STEP 6: EXPLORATORY DATA ANALYSIS
# -----------------------------
# 5A. Missing values plot
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
plt.bar(missing_pct.index, missing_pct.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Missing values (%)")
plt.title("Missing Value Profile")
plt.tight_layout()
plt.savefig(FIG_DIR / "missing_value_profile.png", dpi=300)
# plt.close()

# 5B. Numerical distributions
for col in NUMERICAL_COLS + [TARGET_COL]:
    plt.figure(figsize=(6, 4))
    plt.hist(df[col].dropna(), bins=20)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    safe_name = col.replace(" ", "_").replace("(", "").replace(")", "").replace("%", "percent")
    plt.savefig(FIG_DIR / f"distribution_{safe_name}.png", dpi=300)
    # plt.close()

# 5C. Boxplots of target by categorical variables
for col in CATEGORICAL_COLS:
    temp = df[[col, TARGET_COL]].dropna()
    categories = temp[col].astype(str).unique()
    box_data = [temp.loc[temp[col].astype(str) == c, TARGET_COL].values for c in categories]

    plt.figure(figsize=(max(8, len(categories) * 0.7), 5))
    plt.boxplot(box_data, tick_labels=categories)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(TARGET_COL)
    plt.title(f"{TARGET_COL} across {col}")
    plt.tight_layout()
    safe_name = col.replace(" ", "_").replace("/", "_")
    plt.savefig(FIG_DIR / f"boxplot_{safe_name}.png", dpi=300)
    plt.close()

In [ ]:
# -----------------------------
# STEP 7: CORRELATION MATRIX ANALYSIS
# Only numerical predictors + target
# -----------------------------
corr_df = df[NUMERICAL_COLS + [TARGET_COL]].copy()
corr_matrix = corr_df.corr(method="spearman")
corr_matrix.to_csv(TABLE_DIR / "correlation_matrix.csv")

plt.figure(figsize=(7, 6))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Spearman correlation")
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha="right")
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_matrix.png", dpi=300)
plt.close()

print("\nCorrelation with target:")
print(corr_matrix[TARGET_COL].sort_values(ascending=False))

In [ ]:
# -----------------------------# STEP 7: CORRELATION MATRIX ANALYSIS# Only numerical predictors + target# -----------------------------
import seaborn as sns
corr_df = df[NUMERICAL_COLS + [TARGET_COL]].copy()
corr_matrix = corr_df.corr(method="spearman")
corr_matrix.to_csv(TABLE_DIR / "correlation_matrix.csv")

plt.figure(figsize=(8, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_heatmap.png", dpi=300)
# plt.close()

print("\nCorrelation with target:")
print(corr_matrix[TARGET_COL].sort_values(ascending=False))

In [ ]:
# -----------------------------
# STEP 8: TRAIN-TEST SPLIT
# -----------------------------
X = df[CATEGORICAL_COLS + NUMERICAL_COLS].copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("\nTraining size:", X_train.shape[0])
print("Testing size:", X_test.shape[0])

In [ ]:
# -----------------------------
# STEP 9: PREPROCESSING
# - Median imputation for numeric variables
# - Most frequent imputation for categorical variables
# - Standard scaling for numeric variables
# - One-hot encoding for categorical variables
# -----------------------------
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, NUMERICAL_COLS),
    ("cat", categorical_transformer, CATEGORICAL_COLS)
])


In [ ]:
# -----------------------------
# STEP 10: DEFINE MODELS + GRID SEARCH SPACES
# -----------------------------
models = {
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "SVR": SVR(),
    "MLP": MLPRegressor(random_state=RANDOM_STATE, max_iter=4000)
}

param_grids = {
    "Decision Tree": {
        "model__max_depth": [None, 3, 5, 10, 15],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "Random Forest": {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [None, 5, 10, 15],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2]
    },
    "SVR": {
        "model__kernel": ["rbf", "linear"],
        "model__C": [1, 10, 50, 100],
        "model__epsilon": [0.01, 0.1, 0.2],
        "model__gamma": ["scale", "auto"]
    },
    "MLP": {
        "model__hidden_layer_sizes": [(50,), (100,), (100, 50)],
        "model__activation": ["relu", "tanh"],
        "model__alpha": [0.0001, 0.001, 0.01],
        "model__learning_rate_init": [0.001, 0.01]
    }
}

if HAS_XGB:
    models["XGBoost"] = XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE
    )
    param_grids["XGBoost"] = {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0]
    }


In [ ]:
# -----------------------------
# STEP 11: DEFINE METRICS FUNCTION
# -----------------------------
def get_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred)
    }


In [ ]:
# -----------------------------
# STEP 12: GRID SEARCH + MODEL TRAINING
# -----------------------------
cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

results = []
fitted_models = {}

for model_name, model in models.items():
    print(f"\nRunning Grid Search for: {model_name}")

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[model_name],
        scoring="r2",
        cv=cv,
        n_jobs=-1,
        verbose=0
    )

    grid.fit(X_train, y_train)
    best_estimator = grid.best_estimator_

    # Predictions
    y_train_pred = best_estimator.predict(X_train)
    y_test_pred = best_estimator.predict(X_test)

    # Metrics
    train_metrics = get_metrics(y_train, y_train_pred)
    test_metrics = get_metrics(y_test, y_test_pred)

    results.append({
        "Model": model_name,
        "Best CV R2": grid.best_score_,
        "Train R2": train_metrics["R2"],
        "Test R2": test_metrics["R2"],
        "Train RMSE": train_metrics["RMSE"],
        "Test RMSE": test_metrics["RMSE"],
        "Train MAE": train_metrics["MAE"],
        "Test MAE": test_metrics["MAE"],
        "Best Parameters": str(grid.best_params_)
    })

    fitted_models[model_name] = best_estimator

# Create performance table
results_df = pd.DataFrame(results).sort_values(
    by=["Test R2", "Test RMSE"],
    ascending=[False, True]
).reset_index(drop=True)

print("\nModel performance table:")
print(results_df)

results_df.to_csv(TABLE_DIR / "model_performance_table.csv", index=False)


In [ ]:
# -----------------------------
# STEP 13: PLOT MODEL PERFORMANCE
# -----------------------------
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["Test R2"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Test R²")
plt.title("Model Performance Comparison (Test R²)")
plt.tight_layout()
plt.savefig(FIG_DIR / "model_comparison_test_r2.png", dpi=300)
# plt.close()

plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["Test RMSE"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Test RMSE")
plt.title("Model Performance Comparison (Test RMSE)")
plt.tight_layout()
plt.savefig(FIG_DIR / "model_comparison_test_rmse.png", dpi=300)
# plt.close()

In [ ]:
# -----------------------------
# STEP 14: SELECT THE BEST MODEL
# -----------------------------
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

print("\nBest model:", best_model_name)

In [ ]:
# -----------------------------
# STEP 15: PARITY PLOT (CROSS-PLOT VALIDATION)
# This uses the best model selected after training/tuning.
# -----------------------------
y_train_pred_best = best_model.predict(X_train)
y_test_pred_best = best_model.predict(X_test)

plt.figure(figsize=(7, 7))
plt.scatter(y_train, y_train_pred_best, alpha=0.7, label="Training")
plt.scatter(y_test, y_test_pred_best, alpha=0.7, label="Testing")

min_val = min(y.min(), y_train_pred_best.min(), y_test_pred_best.min())
max_val = max(y.max(), y_train_pred_best.max(), y_test_pred_best.max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Inhibition Efficiency (%)")
plt.ylabel("Predicted Inhibition Efficiency (%)")
plt.title(f"Parity Plot for Inhibition Efficiency Prediction ({best_model_name})")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "parity_plot_best_model.png", dpi=300)
# plt.close()

In [ ]:
# -----------------------------
# STEP 16: WILLIAMS PLOT ANALYSIS
# -----------------------------
def to_dense(X):
    if hasattr(X, "toarray"):
        return X.toarray()
    return np.asarray(X)

def compute_leverage(X_train_t, X_test_t):
    X_train_dense = to_dense(X_train_t)
    X_test_dense = to_dense(X_test_t)

    n_train, p = X_train_dense.shape

    def add_intercept(X):
        return np.column_stack([np.ones(X.shape[0]), X])

    # If transformed feature space is too large, reduce it before leverage calculation
    if p >= (n_train - 1):
        n_components = min(max(2, n_train // 5), 20, n_train - 2)

        if hasattr(X_train_t, "toarray"):
            reducer = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
        else:
            reducer = PCA(n_components=n_components, random_state=RANDOM_STATE)

        X_train_red = reducer.fit_transform(X_train_dense)
        X_test_red = reducer.transform(X_test_dense)

        X_train_design = add_intercept(X_train_red)
        X_test_design = add_intercept(X_test_red)
        p_eff = X_train_red.shape[1]
    else:
        X_train_design = add_intercept(X_train_dense)
        X_test_design = add_intercept(X_test_dense)
        p_eff = p

    xtx_inv = np.linalg.pinv(X_train_design.T @ X_train_design)

    leverage_train = np.sum((X_train_design @ xtx_inv) * X_train_design, axis=1)
    leverage_test = np.sum((X_test_design @ xtx_inv) * X_test_design, axis=1)

    h_star = 3 * (p_eff + 1) / n_train
    return leverage_train, leverage_test, h_star

# Transform the features using the preprocessor from the best model
best_preprocessor = best_model.named_steps["preprocessor"]
X_train_t = best_preprocessor.transform(X_train)
X_test_t = best_preprocessor.transform(X_test)

leverage_train, leverage_test, h_star = compute_leverage(X_train_t, X_test_t)

residual_train = y_train.values - y_train_pred_best
residual_test = y_test.values - y_test_pred_best

std_train = np.std(residual_train, ddof=1)
if std_train == 0:
    std_train = 1e-8

std_resid_train = residual_train / std_train
std_resid_test = residual_test / std_train

plt.figure(figsize=(8, 6))
plt.scatter(leverage_train, std_resid_train, alpha=0.7, label="Training")
plt.scatter(leverage_test, std_resid_test, alpha=0.7, label="Testing")
plt.axhline(3, linestyle="--")
plt.axhline(-3, linestyle="--")
plt.axvline(h_star, linestyle="--")
plt.xlabel("Leverage")
plt.ylabel("Standardized Residual")
plt.title(f"Williams Plot ({best_model_name})")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "williams_plot.png", dpi=300)
# plt.close()

williams_df = pd.DataFrame({
    "Set": ["Training"] * len(leverage_train) + ["Testing"] * len(leverage_test),
    "Leverage": np.concatenate([leverage_train, leverage_test]),
    "Standardized Residual": np.concatenate([std_resid_train, std_resid_test])
})
williams_df.to_csv(TABLE_DIR / "williams_plot_data.csv", index=False)

In [ ]:
# -----------------------------
# STEP 17: SHAP ANALYSIS
# -----------------------------
def get_feature_names(preprocessor):
    feature_names = NUMERICAL_COLS.copy()

    encoder = preprocessor.named_transformers_["cat"].named_steps["encoder"]
    cat_names = encoder.get_feature_names_out(CATEGORICAL_COLS).tolist()

    feature_names.extend(cat_names)
    return feature_names

if HAS_SHAP:
    final_model = best_model.named_steps["model"]
    feature_names = get_feature_names(best_preprocessor)

    X_train_dense = to_dense(X_train_t)
    X_test_dense = to_dense(X_test_t)

    # Faster explainer for tree models
    if best_model_name in ["Decision Tree", "Random Forest", "XGBoost"]:
        explainer = shap.Explainer(final_model, X_train_dense, feature_names=feature_names)
        shap_values = explainer(X_test_dense)
        shap_array = shap_values.values
        shap_plot_data = X_test_dense
    else:
        background = X_train_dense[:min(100, len(X_train_dense))]
        evaluation = X_test_dense[:min(100, len(X_test_dense))]
        explainer = shap.KernelExplainer(final_model.predict, background)
        shap_array = explainer.shap_values(evaluation, nsamples=200)
        shap_plot_data = evaluation

    # SHAP beeswarm plot
    plt.figure()
    shap.summary_plot(shap_array, shap_plot_data, feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "shap_beeswarm.png", dpi=300)
    # plt.close()

    # SHAP bar plot
    plt.figure()
    shap.summary_plot(shap_array, shap_plot_data, feature_names=feature_names, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "shap_bar.png", dpi=300)
    # plt.close()

    shap_importance = pd.DataFrame({
        "Feature": feature_names,
        "MeanAbsSHAP": np.abs(shap_array).mean(axis=0)
    }).sort_values("MeanAbsSHAP", ascending=False)

    shap_importance.to_csv(TABLE_DIR / "shap_feature_importance.csv", index=False)

    print("\nTop SHAP features:")
    print(shap_importance.head(10))
else:
    print("\nSHAP is not installed, so SHAP analysis was skipped.")

In [ ]:
# -----------------------------
# STEP 18: SAVE A CLEAN PERFORMANCE TABLE
# This is the table you can directly use in the paper.
# -----------------------------
performance_table = results_df[[
    "Model",
    "Train R2", "Test R2",
    "Train RMSE", "Test RMSE",
    "Train MAE", "Test MAE"
]].copy()

performance_table.to_csv(TABLE_DIR / "paper_ready_performance_table.csv", index=False)

print("\nPaper-ready performance table:")
print(performance_table)

In [ ]:
# -----------------------------
# STEP 19: OPTIONAL TEXT SUMMARY FOR RESULTS
# -----------------------------
best_row = results_df.iloc[0]
outliers = (np.abs(williams_df["Standardized Residual"]) > 3).sum()
high_leverage = (williams_df["Leverage"] > h_star).sum()

print("\n========== RESULTS SUMMARY ==========")
print(f"Best model: {best_model_name}")
print(f"Best test R²: {best_row['Test R2']:.4f}")
print(f"Best test RMSE: {best_row['Test RMSE']:.4f}")
print(f"Best test MAE: {best_row['Test MAE']:.4f}")
print(f"Williams plot leverage threshold (h*): {h_star:.4f}")
print(f"Points outside ±3 residual range: {outliers}")
print(f"High-leverage points: {high_leverage}")

print(f"\nAll tables saved in: {TABLE_DIR.resolve()}")
print(f"All figures saved in: {FIG_DIR.resolve()}")

In [ ]:
description_df = df.describe()
kurtosis_df = df[NUMERICAL_COLS + [TARGET_COL]].kurt().to_frame(name='kurtosis').T
skewness_df = df[NUMERICAL_COLS + [TARGET_COL]].skew().to_frame(name='skewness').T

# Combine all statistics
full_description_df = pd.concat([description_df, kurtosis_df, skewness_df])
display(full_description_df)